# Week 4 Hand-in: Photometric Redshift Estimation

**Photometric redshifts** (photo-z's) estimate galaxy distances from broadband imaging rather than spectroscopy. A galaxy's spectral energy distribution (SED) shifts redward as it moves away from us; the pattern of fluxes across filters (typically u, g, r, i, z, y for LSST) encodes the redshift. Machine learning regressors can learn the mapping from colors → redshift, but their accuracy depends on several factors we explore here:

| Exercise | Topic |
|----------|-------|
| 1 | Training-set representativeness |
| 2 | Adding near-IR bands (Roman) |
| 3 | Galaxy morphology as an extra feature |
| 4 | Photometric noise and blending contamination |
| 5 | Comparing different photo-z algorithms |
| 6 | Going from per-object p(z) to ensemble n(z) |

The data come from the LSST Photo-z Data Challenge preparation or DP1 studies. Several datasets are split into training and test sets so we can measure out-of-sample performance.

In [ ]:
import tables_io
import utility_functions
import numpy as np
import matplotlib.pyplot as plt
import admixture_functions

#### Download and uncompress the data

In [ ]:
_ = utility_functions.setup_data()

#### Change this to match the correct location

In [ ]:
pz_dir = '.'

## Exercise 1: compare training with representative and non-representative data

The goal of this exercise is to understand the effect of using non-representative data for traning.

At some level this is inevitable when making photometric redshifts: if we could easily get spectra at the same depths to which we can get imaging data and photometry, then we would not need to make photometric redshifts.

To do this we will use files that TQ has prepared extracted from the PZ data challenge preperation.  Specifically both represenative and non-representative training sets.

In [ ]:
# "Representative" training data mirrors the photometric depth/selection of the test set.
# "Non-representative" training data is drawn from a typical spectroscopic survey,
# which is flux-limited and therefore biased toward bright, low-redshift galaxies.
train_rep_data = tables_io.read(f'{pz_dir}/data/task1_representativeness/train_representative_lsstY10_ugrizy_100k.pq')
train_non_rep_data = tables_io.read(f'{pz_dir}/data/task1_representativeness/train_nonrepresentative_specz_lsstY10_ugrizy_100k.pq')
test_data = tables_io.read(f'{pz_dir}/data/task1_representativeness/test_lsstY10_ugrizy_100k.pq')

In [ ]:
# Features: total magnitude (sum of all-band fluxes converted back to mag) + adjacent-band colors
# Colors are the primary photo-z signal; total magnitude helps break flux-color degeneracies.
train_rep_data_targets, train_rep_data_features = utility_functions.prepare_data_total_mag_and_colors(train_rep_data, 'mag_{band}_lsst', 'ugrizy')
train_non_rep_data_targets, train_non_rep_data_features = utility_functions.prepare_data_total_mag_and_colors(train_non_rep_data, 'mag_{band}_lsst', 'ugrizy')
test_targets, test_features = utility_functions.prepare_data_total_mag_and_colors(test_data, 'mag_{band}_lsst', 'ugrizy')

In [ ]:
# Visualize the redshift distributions of each sample.
# A well-matched (representative) training set should overlay the test distribution.
# The non-representative (spectroscopic) set will show a deficit at high redshift
# because faint, distant galaxies are hard to get spectra for.
redshift_bins = np.linspace(0., 2.2, 111)
_ = plt.hist(train_rep_data_targets, bins=redshift_bins, alpha=0.2, label='Train (rep)')
_ = plt.hist(train_non_rep_data_targets, bins=redshift_bins, alpha=0.2, label='Train (non-rep)')
_ = plt.hist(test_targets, bins=redshift_bins, alpha=0.2, label='Test')
_ = plt.legend()
_ = plt.xlabel('z')
_ = plt.ylabel('Objects / 0.02')

### Possible questions to explore:

1. Quantify the overall degradation when using the non-representative training set.
2. Quantify the degredation in specific parts of parameter space when using the non-representative training set.
3. Can a simple resampling can make help emulate the representative training set?

## Exercise 2: explore effects of adding near IR bands to optical bands when making photometric redshifts

The goal of this exercise is to understand the effect of using near IR bands (e.g., from Roman) as well as optical bands.

To do this we will use more files that TQ has prepared extracted from the PZ data challenge preperation.

In [ ]:
# LSST optical-only (ugrizy) vs. optical + Roman near-IR (Y106, J129, H158).
# Near-IR bands extend wavelength coverage to ~1.8 µm, which shifts the Lyman break
# into the NIR at z > 3 and helps break color-redshift degeneracies at intermediate z.
train_optical_data = tables_io.read(f'{pz_dir}/data/task2_nir/train_lsstY10_ugrizy_100k.pq')
train_opt_nir_data = tables_io.read(f'{pz_dir}/data/task2_nir/train_lsstY10_ugrizy_romanYJH_100k.pq')
test_optical_data = tables_io.read(f'{pz_dir}/data/task2_nir/test_lsstY10_ugrizy_100k.pq')
test_opt_nir_data = tables_io.read(f'{pz_dir}/data/task2_nir/test_lsstY10_ugrizy_romanYJH_100k.pq')

In [ ]:
train_opt_nir_data

In [ ]:
# Note the different band name templates: optical bands use 'mag_{band}_lsst',
# but the combined catalog uses 'mag_{band}' with fully-qualified filter names
# (e.g. 'u_lsst', 'Y106_roman') to avoid ambiguity between overlapping bandpasses.
optical_bands = ['u', 'g', 'r', 'i', 'z', 'y']
optical_and_nir_bamds = ['u_lsst', 'g_lsst', 'r_lsst', 'i_lsst', 'z_lsst', 'y_lsst', 'Y106_roman', 'J129_roman', 'H158_roman']

train_optical_targets, train_optical_features = utility_functions.prepare_data_total_mag_and_colors(
    train_optical_data, 'mag_{band}_lsst', optical_bands,
)
test_optical_targets, test_optical_features = utility_functions.prepare_data_total_mag_and_colors(
    test_optical_data, 'mag_{band}_lsst', optical_bands,
)

train_opt_nir_targets, train_optical_features = utility_functions.prepare_data_total_mag_and_colors(
    train_opt_nir_data, 'mag_{band}', optical_and_nir_bamds,
)
test_opt_nir_targets, test_optical_features = utility_functions.prepare_data_total_mag_and_colors(
    test_opt_nir_data, 'mag_{band}', optical_and_nir_bamds,
)

### Possible questions to explore:

1. Quantify the overall improvement when using the near-IR data.
2. Quantify the improvement in specific parts of parameter space when using near-IR data.

## Exercise 3: add morpology information to training features

The goal of this exercise is to see if we can use morphology information to improve on the redshift estimates.

The notebooks, "Extra_Morphology.ipynb" and "Extra_SklearnMorphology.ipynb" can help you explore the possiblities here.

In [ ]:
train = tables_io.read(f'{pz_dir}/data/lsst_cat_matched_nonan_train.hdf5')
test = tables_io.read(f'{pz_dir}/data/lsst_cat_matched_nonan_test.hdf5')


In [ ]:
def make_mask(t):
    # Keep only extended (galaxy) objects with high S/N in four bands.
    # refExtendedness == 1 removes point sources (stars).
    # i_blendedness < 0.01 removes heavily blended objects whose morphology is unreliable.
    # S/N > 10 in g, i, r, z ensures the shape measurements are meaningful.
    mask = (t['refExtendedness'] == 1) *\
           (t['i_blendedness'] < 0.01) *\
           (t['g_psfFlux']/ t['g_psfFluxErr'] > 10) *\
           (t['i_psfFlux'] / t['i_psfFluxErr'] > 10) *\
           (t['r_psfFlux'] / t['r_psfFluxErr'] > 10) *\
           (t['z_psfFlux'] / t['z_psfFluxErr'] > 10)
    return mask

In [ ]:
mask_test = make_mask(test)
mask_train = make_mask(train)

In [ ]:
def get_features(t):
    """ This will include the sersic fit index,    
    the trace in g and z bands, and the trace of the sersic fits
    in addition to the magntidues
    """
    # Sersic profile half-light radius components: n=1 → exponential disk, n=4 → de Vaucouleurs (elliptical)
    sersic_x = t['sersic_reff_x']  
    sersic_y = t['sersic_reff_y']
    # Trace of the Sersic effective radius tensor (encodes galaxy size and ellipticity)
    sersic_trace = np.nan_to_num(sersic_x*sersic_x + sersic_y*sersic_y, nan=0)
    sersic_index = np.nan_to_num(t['sersic_index'], 0)

    # Second moments of the light distribution (shape measurements) in g and z bands;
    # the trace is proportional to the PSF-convolved size squared.
    g_trace = np.nan_to_num(t['g_ixx'] + t['g_iyy'], nan=0)
    z_trace = np.nan_to_num(t['z_ixx'] + t['z_iyy'], nan=0)

    # Stack morphology features + PSF magnitudes into a single feature matrix
    feature_list = [sersic_index, g_trace, z_trace, sersic_trace]
    feature_list += [np.nan_to_num(t[f'{band}_psfMag'], nan=30) for band in 'ugrizy']
    
    return np.vstack(feature_list).T

In [ ]:
features_test = get_features(test)[mask_test]
features_train = get_features(train)[mask_train]

targets_test = test['redshift'][mask_test]
targets_train = train['redshift'][mask_train]

### Possible questions to explore:

1. Can you improve overall p(z) estimation using morphology parameters?
2. Can you improve p(z) estimation in specific parts of the parameters space, e.g., at low redshifts?

## Exercise 4: explore effects of random photometric errors and/or flux contamination from blending

The goal of this exercise is to quantify the effect of both random photometric errors and flux contamination from blending to photo-z estimation.

In [ ]:
train = tables_io.read(f"{pz_dir}/data/dp1_matched_v4_train.hdf5")
test = tables_io.read(f"{pz_dir}/data/dp1_matched_v4_test.hdf5")

In [ ]:
train_features = utility_functions.get_band_values(train, '{band}_gaap1p0Mag', 'ugrizy')
train_targets = train['redshift']
test_features = utility_functions.get_band_values(test, '{band}_gaap1p0Mag', 'ugrizy')
test_targets = test['redshift']

library = utility_functions.get_band_values(test, '{band}_gaap1p0Mag', 'ugrizy')

In [ ]:
# Logarithmically-spaced noise grid from 0.0001 to 1.0 mag (17 points).
# This spans from negligible perturbation to noise comparable to the magnitude itself.
noise_grid = np.logspace(-4, 0, 17)

def make_noised_mags(features):
    the_dict = {}
    inputs = np.hstack([features])

    outputs = []
    for noise in noise_grid:
        # Add the same Gaussian noise level to all 6 bands independently
        noisy_mags = admixture_functions.gaussian_noise(features, noise_levels=noise*np.ones(6)).T
        outputs.append(noisy_mags)
    return outputs

In [ ]:
# Same logarithmic grid for flux admixture fraction.
# admix=0 is no contamination; admix=1 means equal flux from neighbor as from target.
admix_grid = np.logspace(-4, 0, 17)

def make_admix_mags(features):
    the_dict = {}
    inputs = np.hstack([features])
    outputs = []

    for admix in admix_grid:
        # Each object gets a random neighbor from the library; its flux is scaled so
        # the neighbor contributes `admix` fraction of the target's total flux.
        mixed_mags = admixture_functions.make_admixture(train_features, library, admixture=admix)
        outputs.append(mixed_mags)
    return outputs

In [ ]:
noised_mags = make_noised_mags(train_features)
admix_mags = make_admix_mags(train_features)

In [ ]:
# For each noise/admixture level, compute the per-band mean shift and scatter in magnitude.
# This characterizes how much each perturbation type moves galaxies in color space,
# which determines how badly the photo-z estimator will be fooled.
noise_delta = np.zeros((17, 6))
noise_rms = np.zeros((17, 6))
admix_delta = np.zeros((17, 6))
admix_rms = np.zeros((17, 6))

for i in range(17):
    deltas_noise = train_features - noised_mags[i]
    deltas_admix = train_features - admix_mags[i]
    noise_delta[i] = deltas_noise.mean(axis=0)   # mean bias per band
    noise_rms[i] = deltas_noise.std(axis=0)       # scatter per band
    admix_delta[i] = deltas_admix.mean(axis=0)
    admix_rms[i] = deltas_admix.std(axis=0)

In [ ]:
# Plot band-averaged RMS and mean shift vs. perturbation level.
# Noise RMS grows linearly with noise_grid (by construction).
# Admixture introduces both a bias (delta) and scatter;
_ = plt.scatter(noise_grid, noise_rms.mean(axis=1), label='noise scatter')
_ = plt.scatter(admix_grid, admix_rms.mean(axis=1), label='admix scatter')
_ = plt.scatter(admix_grid, admix_delta.mean(axis=1), label='admix bias')
_ = plt.xscale('log')
_ = plt.xlabel('Perturbation level')
_ = plt.ylabel('Mean magnitude shift (mag)')
_ = plt.legend()

### Possible questions to explore:

1. How much are the standard metrics degraded by random photometric noise? By flux contamination?
2. Can you make simple parametric models for now the photo-z gets degraded by these effects?
3. Can you use this simple study to guide data quality cuts we might apply to Rubin data analysis?

## Exercise 5: comparing results from different algorithms

The goal of this exercise is to compare the output of different photo-z algorithms on test data.


For each algorithm, train on the same training set and evaluate on the same test set. Compare using:
- Scatter: σ(Δz / (1+z))
- Bias: mean(Δz / (1+z))
- Outlier fraction: fraction with |Δz / (1+z)| > 0.15

### Possible questions to explore:

1. Which algorithms seem to do best overall?
2. Do certain algorithms do better in particular parts of parameter space?

## Excercise 6: going from per-object p(z) to ensemble n(z)

The goal of this exercise is to go from per-object p(z) to ensemble n(z) distributions.

When a photo-z algorithm produces a probability distribution p(z) per galaxy, the ensemble redshift distribution n(z) can be estimated in several ways:

1. **Stacking**: sum all p(z) curves: n(z) = Σᵢ pᵢ(z)
2. **Point-estimate histogram**: histogram of the peak/mean of each pᵢ(z)
3. **Deconvolution**: more sophisticated methods that correct for p(z) width biases

The stacking method is simple but can be biased if individual p(z) are systematically wrong. Compare the stacked n(z) to the histogram of true redshifts to quantify this bias.

### Possible questions to explore:

1. How well does you algorithm do at bin assignment?
2. How well does you algorithm do at representing the distribution of objects within each bin?

## Hand-in

Due Monday Oct 19 on Canvas. Turn in **this notebook** with: (1) one figure from the exercise you chose; (2) one paragraph, starting with "Exercise N": what did you measure? Quote at least one number read off your figure; (3) the AI-tools line in the last cell filled in. Graded complete/incomplete.

Do the measurement and the plot in your exercise's cell. Then paste the lines that draw the figure into the code cell below, so the figure sits next to the paragraph. Before you upload, choose Run All, save, and download the .ipynb (File → Download). Submit the notebook itself, not a PDF.

**One exercise is the assignment.**  A second exercise is optional and does not change the grade. If you do one, add it in new cells after the AI-tools line.

In [ ]:
# Your hand-in figure: paste the lines that draw it here     

**Exercise N.** Your paragraph here.

**AI tools: none,** or which ones and what for.